# 03 — Rapport par appareil pathologique

Ce notebook génère un rapport HTML pour un appareil donné, avec détail par organe.

**Sortie** : `output/rapport_appareil_<slug>.html`

In [ ]:
# Paramètre papermill
APPAREIL = 'SEIN'

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

import pandas as pd

aphp = pd.read_csv(DATA_DIR / 'aphp_data.csv')
reg  = pd.read_csv(DATA_DIR / 'regional_data.csv')

APPAREILS = sorted(aphp[aphp.appareil != 'TOTAL'].appareil.unique())
assert APPAREIL in APPAREILS, f"Appareil inconnu. Options : {APPAREILS}"

app_aphp = aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == APPAREIL) & (aphp.organe == 'TOTAL')]
print(f"Appareil : {APPAREIL}")
app_aphp

In [ ]:
from chart_utils import (
    line_evolution, donut_market_share, regional_comparison,
    TREATMENT_COLS, GHU_LIST
)

# Tous les GHU + AP-HP pour cet appareil
all_ent = aphp[(aphp.appareil == APPAREIL) & (aphp.organe == 'TOTAL')]

fig = line_evolution(all_ent, 'annee', 'nb_patients', 'entite',
                     f'Patients — {APPAREIL}',
                     entities=['AP-HP'] + GHU_LIST)
fig.show()

In [ ]:
# Donut GHU pour la dernière année
last_year = int(aphp.annee.max())
ghu_last = all_ent[(all_ent.entite.isin(GHU_LIST)) & (all_ent.annee == last_year)]

fig_d = donut_market_share(ghu_last, 'entite', 'nb_patients',
                           f'Répartition GHU — {APPAREIL} ({last_year})')
fig_d.show()

In [ ]:
# Évolution par organe (AP-HP)
organes = aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == APPAREIL) & (aphp.organe != 'TOTAL')]

if len(organes) > 0:
    fig_o = line_evolution(organes, 'annee', 'nb_patients', 'organe',
                           f'Évolution par organe — {APPAREIL}')
    fig_o.show()
else:
    print(f"Pas de sous-organes pour {APPAREIL}")

In [ ]:
# Séjours AP-HP pour cet appareil
melted = app_aphp.melt(
    id_vars=['annee'], value_vars=list(TREATMENT_COLS.keys()),
    var_name='type', value_name='nb_sejours')
melted['label'] = melted['type'].map(TREATMENT_COLS)

fig_s = line_evolution(melted, 'annee', 'nb_sejours', 'label',
                       f'Séjours — {APPAREIL}',
                       entities=list(TREATMENT_COLS.values()))
fig_s.show()

In [ ]:
# Contexte régional
reg_app = reg[(reg.appareil == APPAREIL) & (reg.organe == 'TOTAL')]

if len(reg_app) > 0:
    fig_r = regional_comparison(reg_app, 'nb_patients',
                                f'Contexte régional — {APPAREIL}')
    fig_r.show()

## Génération HTML

In [ ]:
from report_builder import build_rapport_appareil

out = build_rapport_appareil(APPAREIL, DATA_DIR, OUTPUT_DIR)
print(f"Rapport généré : {out}")